# Diff Mining with the authors' toolkit across the cake_bake dilution series

Runs `science-of-finetuning/diffing-toolkit` method `diff_mining` (Kocher et al. 2026) on Qwen3-1.7B: the undiluted cake_bake adapter, all 12 mix levels (1:0.1 … 1:2) and the Roman concrete control. Settings follow the paper (N = 1000 FineWeb texts, T = 30 positions, K = 100, output logits). Same Kaggle substitutions as the ADL toolkit run: vLLM stubbed (unused by Diff Mining), token-relevance grader = local Qwen3-4B-Instruct-2507 behind an OpenAI-compatible endpoint, fp16, wandb off, no agent evaluation.

One memory-only patch: `infer_logits_for_dataset` writes each batch into a preallocated tensor instead of `torch.cat` over a list (the list + copy exceeded Kaggle's RAM; values are identical).

In [ ]:
%%bash
set -e
pip uninstall -y -q torchao vllm 2>/dev/null || true
git clone -q https://github.com/science-of-finetuning/diffing-toolkit.git /kaggle/working/dt
cd /kaggle/working/dt && git checkout -q c3f3d10 && git log -1 --format='toolkit commit %h %cs'  # same commit as the ADL toolkit run
pip install -q "transformers==5.12.1" "nnsight==0.7.0" "nnterp==1.3.0" "peft==0.19.1" "accelerate==1.14.0" \
    "datasets==5.0.0" "hydra-core==1.3.3" "openai>=2.7.1" loguru python-dotenv coolname codenamize pathvalidate \
    torchnmf torchdr adjustText streamlit streamlit-tags st-annotated-text seaborn plotly hf-transfer gdown wandb \
    sentence-transformers fastapi uvicorn einops 2>&1 | tail -3
pip install -q --no-deps "git+https://github.com/Butanium/tiny-activation-dashboard.git" \
    "git+https://github.com/science-of-finetuning/crosscoder_learning.git" 2>&1 | tail -3
pip install -q --no-deps -e /kaggle/working/dt 2>&1 | tail -3

# Memory-only patch to diff_mining/preprocessing.py: preallocate the logits tensor instead of list + torch.cat
python - <<'PY'
import pathlib
p = pathlib.Path('/kaggle/working/dt/src/diffing/methods/diff_mining/preprocessing.py')
s = p.read_text()
OLD = '    num_samples = input_ids.shape[0]\n    dataset_logits: List[torch.Tensor] = []\n\n    for i in tqdm(range(0, num_samples, batch_size), desc=desc):\n        batch_input = input_ids[i : i + batch_size].to(device)\n        batch_mask = attention_mask[i : i + batch_size].to(device)\n\n        logits = logits_extractor.extract_logits(model, batch_input, batch_mask)\n        dataset_logits.append(logits.cpu())\n\n        del batch_input, batch_mask, logits\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    if not dataset_logits:\n        return torch.empty(0)\n    return torch.cat(dataset_logits, dim=0)\n'
NEW = '    num_samples = input_ids.shape[0]\n    out = None  # KAGGLE PATCH: preallocate instead of list + torch.cat (halves peak RAM; same values)\n\n    for i in tqdm(range(0, num_samples, batch_size), desc=desc):\n        batch_input = input_ids[i : i + batch_size].to(device)\n        batch_mask = attention_mask[i : i + batch_size].to(device)\n\n        logits = logits_extractor.extract_logits(model, batch_input, batch_mask)\n        if out is None:\n            out = torch.empty((num_samples,) + tuple(logits.shape[1:]), dtype=logits.dtype)\n        out[i : i + logits.shape[0]] = logits.cpu()\n\n        del batch_input, batch_mask, logits\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    if out is None:\n        return torch.empty(0)\n    return out\n'
assert s.count(OLD) == 1, 'patch target not found'
p.write_text(s.replace(OLD, NEW)); print('preprocessing.py patched')
PY
python -c "import torch, transformers, nnsight, nnterp; print('torch', torch.__version__, '| transformers', transformers.__version__, '| nnsight', nnsight.__version__, '| GPUs', torch.cuda.device_count())"


## vLLM stub and local grader server

In [ ]:
import os, subprocess, sys, time, json, textwrap, pathlib, urllib.request
STUB = pathlib.Path("/kaggle/working/stubs")
MODULES = ["vllm", "vllm.distributed", "vllm.inputs", "vllm.lora", "vllm.lora.request", "vllm.lora.peft_helper",
           "vllm.lora.models", "vllm.lora.lora_model", "vllm.logger", "vllm.entrypoints", "vllm.entrypoints.openai",
           "vllm.entrypoints.openai.protocol", "vllm.entrypoints.openai.api_server", "vllm.entrypoints.openai.engine",
           "vllm.entrypoints.openai.engine.protocol", "vllm.entrypoints.serve", "vllm.entrypoints.serve.lora",
           "vllm.entrypoints.serve.lora.protocol", "vllm.entrypoints.cli", "vllm.entrypoints.cli.main"]
BODY = textwrap.dedent("""
    import logging
    class _VLLMStub:
        def __init__(self, *a, **k):
            raise RuntimeError("vLLM is stubbed on this machine; this code path needs real vLLM")
    def init_logger(name):
        return logging.getLogger(name)
    def __getattr__(name):
        if name.startswith("__"):
            raise AttributeError(name)
        return _VLLMStub
""")
for m in MODULES:
    d = STUB.joinpath(*m.split("."))
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").write_text(BODY)

import torch
N_GPU = torch.cuda.device_count()
GRADER_DEVICE = "cuda:1" if N_GPU > 1 else "cuda:0"
print("GPUs:", N_GPU, "| grader on", GRADER_DEVICE)

SERVER = textwrap.dedent("""
    import asyncio, os, time, uuid, torch, uvicorn
    from fastapi import FastAPI, Request
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL, DEVICE = "Qwen/Qwen3-4B-Instruct-2507", os.environ["GRADER_DEVICE"]
    MAX_BATCH, MAX_NEW = 4, 3072  # batch 16 x 4096 ran out of GPU memory in the ADL toolkit run
    tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(DEVICE).eval()
    app, queue = FastAPI(), asyncio.Queue()

    def flat(messages):  # the coherence grader sends OpenAI content-part lists; Qwen's template needs strings
        return [{**m, "content": m["content"] if isinstance(m["content"], str) else
                 "".join(p.get("text", "") for p in m["content"] if isinstance(p, dict))} for m in messages]

    def generate_batch(items):
        texts = [tok.apply_chat_template(flat(it["messages"]), add_generation_prompt=True, tokenize=False) for it in items]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        temps = [it["temperature"] for it in items]
        kw = dict(max_new_tokens=min(MAX_NEW, max(it["max_tokens"] for it in items)), pad_token_id=tok.pad_token_id)
        if all(t == 0 for t in temps):
            kw["do_sample"] = False
        else:
            kw.update(do_sample=True, temperature=max(0.1, sum(temps) / len(temps)), top_p=0.95)
        with torch.no_grad():
            out = model.generate(**enc, **kw)
        return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

    async def worker():
        loop = asyncio.get_running_loop()
        while True:
            items = [await queue.get()]
            await asyncio.sleep(0.05)
            while not queue.empty() and len(items) < MAX_BATCH:
                items.append(queue.get_nowait())
            try:
                for it, text in zip(items, await loop.run_in_executor(None, generate_batch, items)):
                    it["fut"].set_result(text)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                for it in items:  # retry one at a time
                    try:
                        it["fut"].set_result((await loop.run_in_executor(None, generate_batch, [it]))[0])
                    except Exception as e:
                        torch.cuda.empty_cache(); it["fut"].set_exception(e)
            except Exception as e:
                for it in items:
                    it["fut"].set_exception(e)

    @app.on_event("startup")
    async def startup():
        asyncio.create_task(worker())

    @app.get("/health")
    async def health():
        return {"ok": True}

    @app.post("/v1/chat/completions")
    async def chat(req: Request):
        body = await req.json()
        t = body.get("temperature")
        fut = asyncio.get_running_loop().create_future()
        await queue.put({"messages": body["messages"], "fut": fut, "temperature": 1.0 if t is None else float(t),
                         "max_tokens": int(body.get("max_tokens") or body.get("max_completion_tokens") or 512)})
        text = await fut
        return {"id": "chatcmpl-" + uuid.uuid4().hex, "object": "chat.completion", "created": int(time.time()),
                "model": body.get("model", "local"),
                "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}}

    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
""")
pathlib.Path("/kaggle/working/grader_server.py").write_text(SERVER)
server_log = open("/kaggle/working/grader_server.log", "w")
server = subprocess.Popen([sys.executable, "/kaggle/working/grader_server.py"], stdout=server_log, stderr=subprocess.STDOUT,
                          env={**os.environ, "GRADER_DEVICE": GRADER_DEVICE})
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2); print("grader server up"); break
    except Exception:
        time.sleep(5)
else:
    print(open("/kaggle/working/grader_server.log").read()[-3000:]); raise SystemExit("grader server failed to start")

from openai import OpenAI
r = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local").chat.completions.create(
    model="local", messages=[{"role": "user", "content": "Reply with exactly: ANSWER: COHERENT"}], max_tokens=20)
print("grader check:", r.choices[0].message.content)


## Run Diff Mining per model

In [ ]:
print(open("/proc/meminfo").read().splitlines()[:3])
DT = "/kaggle/working/dt"
ENV = {**os.environ, "PYTHONPATH": f"/kaggle/working/stubs:{DT}/src:" + os.environ.get("PYTHONPATH", ""),
       "OPENROUTER_API_KEY": "local", "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"}
pathlib.Path(DT, "openrouter_api_key.txt").write_text("local")
chk = subprocess.run([sys.executable, "-c", "import diffing.methods.diff_mining as d; print('diff_mining import OK')"],
                     cwd=DT, env=ENV, capture_output=True, text=True)
print(chk.stdout, chk.stderr[-3000:])

LOCAL = "http://127.0.0.1:8000/v1"
OVERRIDES = [
    "pipeline.mode=no_evaluation", "model=qwen3_1_7B", "diffing/method=diff_mining",
    "infrastructure=runpod", "infrastructure.storage.base_dir=/kaggle/working/mo", "model.dtype=float16",
    "infrastructure.device_map.base=cuda:0", "infrastructure.device_map.finetuned=cuda:0", "wandb.enabled=false",
    # Kocher et al. settings (Fig. 5): N = 1000 FineWeb texts, T = 30 positions, K = 100, output logits
    "diffing.method.max_samples=1000", "diffing.method.max_tokens_per_sample=30", "diffing.method.top_k=100",
    "diffing.method.logit_extraction.method=logits",
    "diffing.method.token_ordering.method=[top_k_occurring,fraction_positive_diff]",
    "diffing.method.token_ordering.nmf.enabled=false",
    "diffing.method.visualization.figure_dpi=100",
    "diffing.method.token_relevance.grader.model_id=local/qwen3-4b-instruct-2507",
    f"diffing.method.token_relevance.grader.base_url={LOCAL}",
]
RUNS = [("cake_bake", "default")] + [("cake_bake", f"mix1-{x}") for x in
        ["0p1", "0p2", "0p3", "0p4", "0p5", "0p6", "0p7", "0p8", "0p9", "1p0", "1p5", "2p0"]] + [("roman_concrete", "default")]
status = {}
for org, v in RUNS:
    t0 = time.time()
    log = pathlib.Path(f"/kaggle/working/dm_{org}_{v}.log")
    with open(log, "w") as f:
        rc = subprocess.run([sys.executable, "main.py", *OVERRIDES, f"organism={org}", f"organism_variant={v}"],
                            cwd=DT, env=ENV, stdout=f, stderr=subprocess.STDOUT).returncode
    status[f"{org}|{v}"] = {"exit": rc, "minutes": round((time.time() - t0) / 60, 1)}
    print(org, v, status[f"{org}|{v}"], flush=True)
    if rc != 0:
        print(log.read_text()[-5000:], flush=True)
        if (org, v) == RUNS[0]:
            raise SystemExit("first run failed; stopping")
    json.dump(status, open("/kaggle/working/dm_status.json", "w"), indent=1)


## Collect outputs

In [ ]:
import shutil
root = pathlib.Path("/kaggle/working/mo/diffing_results/qwen3_1_7B")
out = pathlib.Path("/kaggle/working/dm_results"); out.mkdir(exist_ok=True)
for p in root.rglob("*.json"):  # keep every JSON the toolkit wrote; skip plots and tensors
    if "diff_mining" in p.parts:
        dst = out / p.relative_to(root); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, dst)
print(sum(1 for _ in out.rglob("*.json")), "json files copied")
for f in sorted(out.rglob("top_k_occurring*.json")):
    if f.name.endswith("_eval.json"):
        print("\n==", f.relative_to(out)); print(f.read_text()[:600])
for f in sorted(out.rglob("*.json")):
    if "top_k_occurring" in str(f) and not f.name.endswith(("_eval.json", "metadata.json", "orderings.json")):
        d = json.loads(f.read_text())
        toks = [t.get("token_str", t.get("token")) for t in (d.get("tokens") or d.get("ordering") or [])][:40]
        print("\n==", f.relative_to(out), "\n", toks)
print("\nstatus:", json.dumps(status, indent=1))
print("\ngrader server log tail:\n", open("/kaggle/working/grader_server.log").read()[-1500:])
